<a href="https://colab.research.google.com/github/YuraLashenko/Laboratornaya2/blob/main/%D0%9B%D1%8F%D1%88%D0%B5%D0%BD%D0%BA%D0%BE_%D0%AE%D1%80%D0%B0_%D0%9F%D0%B8%D1%8D_51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Данные (ID, phone).xlsx to Данные (ID, phone).xlsx
Saving Данные (ФИО).xlsx to Данные (ФИО).xlsx
Saving Текстовые сообщения.xlsx to Текстовые сообщения.xlsx


In [ ]:
import pandas

df = pandas.read_excel("Текстовые сообщения.xlsx", header = None)
# for i in range(len(df)):
#   print(df.iloc[i , 0])

def getLineCountInFile():
  print("Количество строк в файле: " + str(len(df)))
  return len(df)

getLineCountInFile()

Количество строк в файле: 203


203

In [ ]:
def getSymbolCountOnLine():
  for i in range(len(df)):
    tempCount = 0
    for j in range(len(str(df.iloc[i , 0]))):
      tempCount += 1
    print(f"В строке №{i + 1} находится {tempCount} символов")

getSymbolCountOnLine()

In [ ]:
def getSymbolCountByType():

  vovelLetters = "уеыаоэяию"
  consonantLetters = "йцкнгшщзхъфвпрлджчсмтьб"
  punctuationMarks = ",.:;!?"
  vovelCount, consonantCount, markCount = 0 , 0 , 0

  for i in range(len(df)):
    for symbol in str(df.iloc[i , 0]):
      if symbol.lower() in vovelLetters:
        vovelCount += 1
      elif symbol.lower() in consonantLetters:
        consonantCount += 1
      elif symbol.lower() in punctuationMarks:
        markCount += 1
  print(f"В файле найдено:\nСогласных: {consonantCount}\nГласных: {vovelCount}\nЗнаков препинания: {markCount}")

getSymbolCountByType()


В файле найдено:
Согласных: 6141
Гласных: 4999
Знаков препинания: 159


In [ ]:
def getClearPhoneNumber(numbersList):

  clearNumber = None
  for number in numbersList:
    if len(number) == 10 or len(number) == 11:
      clearNumber = number

  if clearNumber:
    if len(clearNumber) != 11:
      clearNumber = '7' + clearNumber

    if clearNumber[0] == '8':
      clearNumber = '7' + clearNumber[1:]

    clearNumber = '+' + clearNumber[0] + ' (' + clearNumber[1:4] + ') ' + clearNumber[4:7] + ' ' + clearNumber[7:9] + '-' + clearNumber[9:11]

  if clearNumber != None:
    return clearNumber
  else: return "Номер не указан" # +7 (904) 989 88-50


def searchPhoneNumberInLine():

  numbers = "0123456789"
  numberDictionary = {
      'Номера' : []
  }


  for i in range(len(df)):
    tempNumber = ''
    numberList = []
    Lenght = str(df.iloc[i , 0])
    for symbol in range(len(Lenght)):
      if Lenght[symbol] in numbers:
        tempNumber += Lenght[symbol]
      if symbol + 1 < len(Lenght):
        if Lenght[symbol + 1] not in numbers and Lenght[symbol] not in numbers:
          if tempNumber != '':
            numberList.append(tempNumber)
          tempNumber = ''

    if tempNumber != '':
      numberList.append(tempNumber)

    numberDictionary['Номера'].append(getClearPhoneNumber(numberList))

  return numberDictionary

peopleNumbres = searchPhoneNumberInLine()

saveFile = pandas.DataFrame(peopleNumbres)
saveFile.to_csv('Numbers.csv', index=False , encoding='utf-8-sig')



In [ ]:
from datetime import date

fileFIO = pandas.read_excel('Данные (ФИО).xlsx')
filePHONE = pandas.read_excel('Данные (ID, phone).xlsx')

dataBase = {
    'ID': [],
    'ФИО': [],
    'Номер': [],
    'Пол': [],
    'Возраст': []
}

def getClientName(line):

  name = ''

  for word in str(line).split():
    if word[0].isupper():
      if word.lower() != 'прошу':
        name += word + ' '

  return name

def getAge(dateBirthey):

  age = None

  age = date.today().year - dateBirthey.date().year
  return age

def getGender(name, isPandas):

  gender = None
  name = name.lower()

  if name.endswith('ич'):
    gender = 'М'
  elif name.endswith('на') or name.endswith('ва'):
    gender = 'Ж'
  else:
    if isPandas: gender = fileFIO.loc[i , 'Пол']
    else: gender = 'Данные не найдены'

  return gender

for i in range(len(fileFIO)):
  for j in range(len(filePHONE)):
    if fileFIO.loc[i, 'ID'] == filePHONE.loc[j , 'ID']:
      dataBase['ID'].append(fileFIO.loc[i, 'ID'])

      if pandas.notna(fileFIO.loc[i,'ФИО']):
        dataBase['ФИО'].append(fileFIO.loc[i,'ФИО'])
        dataBase['Пол'].append(getGender(fileFIO.loc[i,'ФИО'] , True))
        dataBase['Возраст'].append(getAge(fileFIO.loc[i,'Дата рождения']))

      else:
        dataBase['ФИО'].append("Данные не найдены")
        dataBase['Пол'].append("Данные не найдены")
        dataBase['Возраст'].append("Данные не найдены")

      if pandas.notna(filePHONE.loc[j,'Номер телефона']):
        dataBase['Номер'].append(getClearPhoneNumber( [str(filePHONE.loc[j,'Номер телефона']) ] ))
      else: dataBase['Номер'].append("Данные не найдены")

for i in range(len(dataBase['Номер'])):
  if dataBase['ФИО'][i] == 'Данные не найдены' and dataBase['Номер'][i] != 'Данные не найдены':
    for j in peopleNumbres['Номера']:
      if dataBase['Номер'][i] == j:
        dataBase['ФИО'][i] = getClientName(df.iloc[peopleNumbres['Номера'].index(j), 0])
        # print(dataBase['ФИО'][i])
        dataBase['Пол'][i] = getGender(dataBase['ФИО'][i][:-1] , False)


db = pandas.DataFrame(dataBase)
db.to_csv('List.csv', index=False , encoding='utf-8-sig')




In [ ]:
def createStatistic():

  file = pandas.read_csv('List.csv')

  file['Возраст'] = pandas.to_numeric(file['Возраст'] , errors='coerce')
  file = file.sort_values(by='Возраст', ascending=False )

  menCount = 0
  womenCount = 0
  noGenderCount = 0

  for i in range(len(file)):
    if file.loc[i , 'Пол'] == "М":
      menCount += 1
    elif  file.loc[i , 'Пол'] == "Ж":
      womenCount += 1
    else: noGenderCount += 1

  print(f'Число мужчин {menCount}\nЧисло женщин {womenCount}\nНеопознано {noGenderCount}')

  pandas.DataFrame(file).to_csv("Statistica.csv" , index=False , encoding='utf-8-sig')

createStatistic()

Число мужчин 16
Число женщин 134
Неопознано 44
